In [11]:
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as md

def crawl_webpage_to_md(url, output_file, content_selector=None):
    try:
        # Giả lập User-Agent để tránh bị block bởi một số server
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
        }

        print(f"Đang tải dữ liệu từ: {url}")
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parse HTML
        soup = BeautifulSoup(response.text, 'html.parser')
        element = soup.select_one(content_selector)

        # # Dọn dẹp bớt các thẻ rác thường gặp nếu không dùng selector cụ thể
        # for element in soup(['script', 'style', 'nav', 'footer', 'noscript', 'iframe']):
        #     element.decompose()

        # # Trích xuất nội dung dựa trên CSS Selector
        # if content_selector:
        #     main_content = soup.select_one(content_selector)
        #     if not main_content:
        #         print(f"Lỗi: Không tìm thấy nội dung với selector '{content_selector}'")
        #         return
        #     html_content = str(main_content)
        # else:
        #     # Nếu không có selector, lấy toàn bộ thẻ body
        #     html_content = str(soup.body) if soup.body else str(soup)

        # Chuyển đổi HTML sang Markdown
        # print("Đang chuyển đổi sang định dạng Markdown...")
        # markdown_text = md(html_content, heading_style="ATX", strip=['img', 'a'])
        # Xóa strip=['img', 'a'] nếu bạn muốn giữ lại hình ảnh và link

        # Ghi ra file .md
         # Bước 1: Xoá thẻ nhiễu trước khi convert
        for noise in element.select(', '.join(_NOISE_TAGS)):
            noise.decompose()

        # Bước 2: HTML → Markdown
        raw_md = md_convert(
            str(element),
            heading_style='ATX',        # dùng # ## ### thay vì gạch dưới
            bullets='-',                # danh sách dùng -
            autolinks=True,             # URL trần → <https://…>
            newline_style='backslash',  # <br> → \ (sạch hơn 2 spaces)
            convert_charrefs=True,      # &amp; → &, &nbsp; → space, v.v.
        )

        # Bước 3: Dọn khoảng trắng thừa
        raw_md = re.sub(r'[ \t]+$', '', raw_md, flags=re.MULTILINE)  # trailing spaces
        raw_md = re.sub(r'\n{3,}', '\n\n', raw_md)                   # tối đa 1 dòng trống
        raw_md = raw_md.strip()

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(raw_md)

        # with open(output_file, 'w', encoding='utf-8') as f:
        #     f.write(markdown_text)

        print(f"Thành công! Đã lưu file tại: {output_file}")

    except requests.exceptions.RequestException as e:
        print(f"Lỗi kết nối mạng/URL: {e}")
    except Exception as e:
        print(f"Có lỗi hệ thống xảy ra: {e}")


# --- CÁCH SỬ DỤNG ---
def fmain():
    # URL mục tiêu
    target_url = "https://phatphapungdung.com/phap-bao/kinh-tieu-bo-thich-minh-chau-111811.html"

    # CSS selector trỏ tới thẻ div/article chứa nội dung chính
    # Thường là '.post-content', '.entry-content', hoặc 'article'
    # Phải inspect (F12) trên trình duyệt để lấy selector chính xác
    target_selector = ".td-post-content"

    output_filename = "0.Kinh_Tieu_Bo.md"

    crawl_webpage_to_md(
        url=target_url,
        output_file=output_filename,
        content_selector=target_selector
    )
fmain()

Đang tải dữ liệu từ: https://phatphapungdung.com/phap-bao/kinh-tieu-bo-thich-minh-chau-111811.html
Thành công! Đã lưu file tại: 0.Kinh_Tieu_Bo.md


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def extract_links_to_md(url, content_selector, output_file):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }

    try:
        print(f"Đang tải trang: {url}")
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Tìm vùng chứa nội dung chính
        main_content = soup.select_one(content_selector)
        if not main_content:
            print(f"Lỗi: Không tìm thấy vùng nội dung với selector '{content_selector}'")
            return

        # Lọc và lấy các link hợp lệ
        markdown_links = []
        seen_urls = set()

        for a_tag in main_content.find_all('a', href=True):
            href = a_tag['href']

            # Bỏ qua link nội bộ trang (mỏ neo) hoặc link rỗng
            if href.startswith('#') or href.startswith('javascript:'):
                continue

            # Chuyển đổi link tương đối thành tuyệt đối
            absolute_url = urljoin(url, href)

            # Tránh trùng lặp link
            if absolute_url in seen_urls:
                continue
            seen_urls.add(absolute_url)

            # Lấy tiêu đề hiển thị của link
            title = a_tag.get_text(strip=True)
            if not title:
                title = "Xem chi tiết bài viết"

            # Định dạng theo chuẩn Markdown: - [Tiêu đề](URL)
            markdown_links.append(f"- [{title}]({absolute_url})")

        # Ghi danh sách link vào file .md
        if markdown_links:
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(f"# Danh sách bài viết trích xuất từ {url}\n\n")
                f.write("\n".join(markdown_links))
            print(f"Thành công! Đã lưu {len(markdown_links)} link vào file: {output_file}")
        else:
            print("Không tìm thấy đường link nào trong vùng nội dung được chọn.")

    except Exception as e:
        print(f"Có lỗi xảy ra: {e}")

# --- THỰC THI ---
def fmain():
    target_url = "https://phatphapungdung.com/phap-bao/kinh-tieu-bo-thich-minh-chau-111811.html"

    # Selector bao quanh khu vực mục lục bài viết
    # Với trang này, bạn có thể dùng '.entry-content' hoặc '.post-content'
    target_selector = ".td-post-content"

    output_filename = "danh_sach_links.md"

    extract_links_to_md(target_url, target_selector, output_filename)
fmain()

Đang tải trang: https://phatphapungdung.com/phap-bao/kinh-tieu-bo-thich-minh-chau-111811.html
Thành công! Đã lưu 139 link vào file: danh_sach_links.md


In [12]:
# crawl content

import os
import re
import requests
from bs4 import BeautifulSoup
from slugify import slugify
from markdownify import markdownify as md_convert

# Đường dẫn tới file markdown chứa danh sách URL của bạn
INPUT_FILE = '0.danh_sach_links.md'

def extract_links_from_md(file_path):
    """Đọc file .md và trích xuất tên cùng URL bằng Regex"""
    links = []
    # Regex tìm cấu trúc - [TÊN](URL)
    pattern = re.compile(r'-\s+\[(.*?)\]\((.*?)\)')

    if not os.path.exists(file_path):
        print(f"Không tìm thấy file: {file_path}")
        return links

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            match = pattern.search(line)
            if match:
                name, url = match.groups()
                links.append((name.strip(), url.strip()))
    return links


_NOISE_TAGS = [
    'script', 'style', 'nav', 'footer',
    'iframe', 'noscript', 'aside', 'button',
    'form', 'input', 'select', 'textarea',
]

def crawl_content(url: str, selector: str) -> str | None:
    """
    Crawl nội dung từ URL theo CSS selector.
    Trả về Markdown giữ nguyên heading, link, bảng, code block, danh sách.
    """
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        )
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8'

        if response.status_code != 200:
            print(f"Lỗi tải trang ({response.status_code}): {url}")
            return None

        soup = BeautifulSoup(response.text, 'html.parser')
        element = soup.select_one(selector)

        if element is None:
            print(f"Không tìm thấy selector '{selector}' tại: {url}")
            return None

        # Bước 1: Xoá thẻ nhiễu trước khi convert
        for noise in element.select(', '.join(_NOISE_TAGS)):
            noise.decompose()

        # Bước 2: HTML → Markdown
        raw_md = md_convert(
            str(element),
            heading_style='ATX',        # dùng # ## ### thay vì gạch dưới
            bullets='-',                # danh sách dùng -
            autolinks=True,             # URL trần → <https://…>
            newline_style='backslash',  # <br> → \ (sạch hơn 2 spaces)
            convert_charrefs=True,      # &amp; → &, &nbsp; → space, v.v.
        )

        # Bước 3: Dọn khoảng trắng thừa
        raw_md = re.sub(r'[ \t]+$', '', raw_md, flags=re.MULTILINE)  # trailing spaces
        raw_md = re.sub(r'\n{3,}', '\n\n', raw_md)                   # tối đa 1 dòng trống
        raw_md = raw_md.strip()

        return raw_md

    except Exception as e:
        print(f"Lỗi khi crawl {url}: {e}")
        return None



def fmain():
    links = extract_links_from_md(INPUT_FILE)
    if not links:
        print("Danh sách URL trống hoặc file không tồn tại.")
        return

    output_dir = 'output_kinh'
    os.makedirs(output_dir, exist_ok=True)

    index_entries = []  # Tích lũy để ghi file kết quả

    for i, (name, url) in enumerate(links, start=1):
        file_slug = slugify(name)
        file_name = f"kn-{i:03d}-{file_slug}.md"          # ← kn-001-...
        file_path = os.path.join(output_dir, file_name)

        print(f"[{i:03d}] Đang crawl: {name}...")
        content = crawl_content(url, selector='.td-main-content')

        if content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(f"# {name}\n\n")
                f.write(content)
            print(f"     -> Đã lưu: {file_path}")
            index_entries.append(f"- [{name}]({file_name})")   # ← dùng file_name, không phải url
        else:
            print(f"     -> Thất bại: {name}")
            index_entries.append(f"- [{name}]({url})  <!-- crawl thất bại -->")

    # Ghi file index
    index_path = os.path.join(output_dir, '_index.md')
    with open(index_path, 'w', encoding='utf-8') as f:
        f.write("# Danh sách kinh\n\n")
        f.write('\n'.join(index_entries))
    print(f"\nĐã tạo file index: {index_path}")

fmain()

# def fmain():
#     # 1. Lấy danh sách link từ file markdown
#     links = extract_links_from_md(INPUT_FILE)
#     if not links:
#         print("Danh sách URL trống hoặc file không tồn tại.")
#         return

#     # Tạo thư mục lưu kết quả nếu chưa có
#     output_dir = 'output_kinh'
#     os.makedirs(output_dir, exist_ok=True)

#     # 2. Lặp qua từng link để crawl và lưu file
#     for name, url in links:
#         # Chuyển đổi tên thành định dạng slug tiếng Việt không dấu
#         # Ví dụ: "TẬP 1 – KINH TIỂU TỤNG" -> "tap-1-kinh-tieu-tung"
#         file_slug = slugify(name)
#         file_name = f"{file_slug}.md"
#         file_path = os.path.join(output_dir, file_name)

#         print(f"Đang crawl: {name}...")
#         content = crawl_content(url, selector='.td-main-content')

#         if content:
#             # Ghi nội dung vào file markdown mới
#             with open(file_path, 'w', encoding='utf-8') as f:
#                 f.write(f"# {name}\n\n")
#                 f.write(content)
#             print(f"-> Đã lưu: {file_path}")
#         else:
#             print(f"-> Thất bại: {name}")

# fmain()

[001] Đang crawl: TẬP 1 – KINH TIỂU TỤNG...
     -> Đã lưu: output_kinh/kn-001-tap-1-kinh-tieu-tung.md
[002] Đang crawl: TẬP 2 – KINH PHÁP CÚ...
     -> Đã lưu: output_kinh/kn-002-tap-2-kinh-phap-cu.md
[003] Đang crawl: TẬP 3 – KINH PHẬT TỰ THUYẾT...
     -> Đã lưu: output_kinh/kn-003-tap-3-kinh-phat-tu-thuyet.md
[004] Đang crawl: Chương 1: Phẩm Bồ Ðề...
     -> Đã lưu: output_kinh/kn-004-chuong-1-pham-bo-de.md
[005] Đang crawl: Chương 5: Phẩm Trưởng Lão Sona...
     -> Đã lưu: output_kinh/kn-005-chuong-5-pham-truong-lao-sona.md
[006] Đang crawl: Chương 2: Phẩm Muccalinda...
     -> Đã lưu: output_kinh/kn-006-chuong-2-pham-muccalinda.md
[007] Đang crawl: Chương 6: Phẩm Sanh Ra Ðã Mù...
     -> Đã lưu: output_kinh/kn-007-chuong-6-pham-sanh-ra-da-mu.md
[008] Đang crawl: Chương 3: Phẩm Nanda...
     -> Đã lưu: output_kinh/kn-008-chuong-3-pham-nanda.md
[009] Đang crawl: Chương 7: Phẩm Nhỏ...
     -> Đã lưu: output_kinh/kn-009-chuong-7-pham-nho.md
[010] Đang crawl: Chương 4: Phẩm Meghiya...

In [2]:
from pathlib import Path
import re

folder = Path("../../docs/kinhtieubo/thichminhchau/x")

for file in folder.glob("*.md"):
    old_name = file.name

    # kn-071-phan-1-chuong-i.md
    new_name = re.sub(
        r"^(kn-\d{3}-)",
        r"\1tap-10-",
        old_name
    )

    if new_name != old_name:
        file.rename(file.with_name(new_name))
        print(f"{old_name} -> {new_name}")

kn-107-09-chuong-iv-pham-bon-bai-ke-tt.md -> kn-107-tap-10-09-chuong-iv-pham-bon-bai-ke-tt.md
kn-087-phan-2-chuong-2.md -> kn-087-tap-10-phan-2-chuong-2.md
kn-091-04-pham-asadisa.md -> kn-091-tap-10-04-pham-asadisa.md
kn-133-06-chuong-xxii-dai-pham-4.md -> kn-133-tap-10-06-chuong-xxii-dai-pham-4.md
kn-114-05-chuong-ix-pham-chin-bai-ke.md -> kn-114-tap-10-05-chuong-ix-pham-chin-bai-ke.md
kn-098-phan-3-chuong-iii-chuong-iv.md -> kn-098-tap-10-phan-3-chuong-iii-chuong-iv.md
kn-077-06-pham-asimsa.md -> kn-077-tap-10-06-pham-asimsa.md
kn-102-04-chuong-iii-pham-ba-bai-ke-tt.md -> kn-102-tap-10-04-chuong-iii-pham-ba-bai-ke-tt.md
kn-095-08-pham-kasava.md -> kn-095-tap-10-08-pham-kasava.md
kn-129-02-chuong-xxi-pham-tam-muoi-bai-ke-tt.md -> kn-129-tap-10-02-chuong-xxi-pham-tam-muoi-bai-ke-tt.md
kn-121-03-chuong-xv-pham-hai-muoi-bai-ke.md -> kn-121-tap-10-03-chuong-xv-pham-hai-muoi-bai-ke.md
kn-075-04-pham-kulavaka.md -> kn-075-tap-10-04-pham-kulavaka.md
kn-092-05-pham-ruhaka.md -> kn-092-tap-10-